# Cross Validation Classification

In [ ]:
import sklearn as sk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Importing data

In [ ]:
df = pd.read_parquet("advanced_processed_data.parquet")

## Preprocessing

In [ ]:
data = df.copy().drop("Severity", axis = 1)
target = df.copy()["Severity"]

### Helper Functions

In [ ]:
def print_results(fold, y_test,prediction):
    confusion = sk.metrics.confusion_matrix(y_test,prediction)
    accuracy = accuracy_score(y_test,prediction)
    precision, recall, f1, support = precision_recall_fscore_support(y_test,prediction)

    print(f"fold: {fold}")
    print("Confusion Matrix")
    print(confusion)
    print(f"Accuracy: {accuracy}:.3f")

    for i, severity in enumerate([0,1,2,3]):
        print(f"Severity: {severity}:",
            f"Precision: {precision[i]:.3f}",
            f"Recall: {recall[i]:.3f}",
            f"F1: {f1[i]:.3f}")

    return confusion, accuracy, precision, recall, f1, support

def collect_details(model, data, target, skf):
    rows = []
    confusions = []

    for fold, (train_index, test_index) in enumerate(skf.split(data, target), start = 1):
        X_train, X_test = data.iloc[train_index], data.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]

        model.fit(X_train,y_train)
        prediction = model.predict(X_test)

        confusion, accuracy, precision, recall, f1, support = print_results(fold, y_test, prediction)

        confusions.append(confusion)

        for i, severity in enumerate(target.unique()):
            rows.append({
                "Fold": fold,
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "F1_score": f1,
                "Support": support
            })

    return pd.DataFrame(rows), confusions

## 10 Fold Stratified Cross Validation

In [ ]:
skf = sk.model_selection.StratifiedKFold(n_splits=10)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

### Dummy Classifier

In [ ]:
dummy = sk.dummy.DummyClassifier(strategy="most_frequent")
dummy_cv = sk.model_selection.cross_validate(dummy,data,target,cv=skf,scoring,return_train_score=False)